# Burgers' equation in 1D

## Setting up the problem

The one-dimensional Burgers' equation with homogeneous Dirichlet boundary conditions is given by
$$\frac{\partial u}{\partial t}+u\frac{\partial u}{\partial x}=D\frac{\partial^2 u}{\partial x^2},\quad \text{for}\quad x\in (0,L)\quad \text{and}\quad t>0,$$
$$u(0,t)=0,\quad u(L,t)=0,$$
$$u(x,0)=g(x).$$

Below, we choose the diffusion coefficient $D$, length of the domain $L$, and initial condition $g(x)$. 

In [1]:
using Mantis
using GLMakie

# Diffusion coefficient
const D = 0.5

# Length of the domain
L = 1.0

# Initial condition
g_expression(x) = [@. sin(π/L*x[:, 1])];

## Defining the finite element space

Now, we choose the number of elements to divide the domain into, and the polynomial degree and smoothness of the basis functions, to create a piecewise-polynomial function space $V_h$.

In [2]:
# The number of elements in the mesh
num_elements = 25
line_geometry = Geometry.create_cartesian_box((0.0,), (L,), (num_elements,))

# The degree of the piecewise-polynomial basis functions
polynomial_degree = 2

# The smoothness of the basis functions (-1 <= smoothness <= p-1)
smoothness = 1

# The piecewise-polynomial function space
V_h = FunctionSpaces.BSplineSpace(line_geometry, polynomial_degree, smoothness)

# Use this function space as a differential form
V⁰ = Forms.FormSpace(0, V_h, L"V^0")

# Use the expressions as form fields
g = Forms.AnalyticalFormField(0, g_expression, line_geometry, "g");

## Deriving the weak form and assembling the system of ODEs

We define the trial and test space, respectively, as
$$S_h:=\{v_h(x)\in V_h\ :\ v_h(0)=0,\ v_h(L)=0\},$$
$$W_h:=S_h.$$

The weak problem is then: find $u_h(x,t)\in S_h$ at fixed time $t$ such that
$$\int_0^L\frac{\partial u_h}{\partial t}w_hdx+\int_0^L\frac{\partial}{\partial x}(\frac{1}{2}u_h^2)w_hdx-D\int_0^L\frac{\partial^2 u_h}{\partial x^2}w_hdx=0,\quad \forall w_h\in W_h.$$

Using integration by parts and the homogeneous boundary conditions of $w_h$ gives
$$\int_0^L\frac{\partial u_h}{\partial t}w_hdx-\frac{1}{2}\int_0^Lu_h^2\frac{\partial w_h}{\partial x}dx+D\int_0^L\frac{\partial u_h}{\partial x}\frac{\partial w_h}{\partial x}dx=0,\quad \forall w_h\in W_h.$$

Let $\phi_i(x),\ i=1,\ldots,n$ form a basis for $V_h$. Then we can write $u_h(x,t)=\sum_{j=1}^nc_j(t)\phi_j(x)$. Since all $w_h\in W_h$ are also linear combinations of the $\phi_i$, the following $n$ equations must be satisfied:
$$\int_0^L\frac{\partial}{\partial t}(\sum_{j=1}^nc_j\phi_j)\phi_idx-\frac{1}{2}\int_0^L(\sum_{j=1}^nc_j\phi_j)^2\frac{d\phi_i}{dx}dx+D\int_0^L\frac{\partial}{\partial x}(\sum_{j=1}^nc_j\phi_j)\frac{d\phi_i}{dx}dx=0,\quad i=1\ldots,n$$
$$\iff \sum_{j=1}^n\frac{dc_j}{dt}\int_0^L\phi_j\phi_idx-\frac{1}{2}\int_0^L(\sum_{j=1}^nc_j\phi_j)^2\frac{d\phi_i}{dx}dx+D\sum_{j=1}^nc_j\int_0^L\frac{d\phi_j}{dx}\frac{d\phi_i}{dx}dx=0,\quad i=1\ldots,n.$$

We can write this system of equations as
$$\mathbf{M}\frac{d\mathbf{C}}{dt}+\mathbf{F}(\mathbf{C})+\mathbf{KC}=\mathbf{0},$$
with:
-  $\mathbf{M}_{ij}=\int_0^L\phi_j\phi_idx,$
-  $\mathbf{F}_i(\mathbf{C})=-\frac{1}{2}\int_0^L(\sum_{j=1}^nc_j\phi_j)^2\frac{d\phi_i}{dx}dx,$
-  $\mathbf{K}_{ij}=D\int_0^L\frac{d\phi_j}{dx}\frac{d\phi_i}{dx}dx.$

In [3]:
# Function to assemble the system matrices
function assemble_system_matrices(
    V::Forms.FormSpace,
    D::Float64,
    dΩ::Quadrature.AbstractQuadratureRule,
)
    
    weak_form_inputs = Assemblers.WeakFormInputs(V, V)
    bc = Forms.set_dirichlet_boundary_conditions(V, 0.0)

    w⁰ = Assemblers.get_test_form(weak_form_inputs)
    u⁰ = Assemblers.get_trial_form(weak_form_inputs)

    M = ∫(w⁰ ∧ ★(u⁰), dΩ)
    lhs_expressions = ((M,),)

    rhs_expressions = ((0,),)

    weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
    M, _ = Assemblers.assemble(weak_form, bc)

    K = ∫(d(w⁰) ∧ ★(d(u⁰)), dΩ)
    lhs_expressions = ((K,),)

    weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
    K, _ = Assemblers.assemble(weak_form, bc)

    return M, D.*K
end

# Define the quadrature
quadrature_degree = polynomial_degree + 2
quadrature_rule = Quadrature.gauss_legendre(quadrature_degree)
dΩ = Quadrature.StandardQuadrature(quadrature_rule, num_elements)

# Assemble the system matrices
M, K = assemble_system_matrices(V⁰, D, dΩ);


In [ ]:
function assemble_nonlinear_term(
    V::Forms.FormSpace,
    u_h::Forms.FormField,
    dΩ::Quadrature.AbstractQuadratureRule,
)
    
    weak_form_inputs = Assemblers.WeakFormInputs(V, V)
    bc = Forms.set_dirichlet_boundary_conditions(V, 0.0)

    w⁰ = Assemblers.get_test_form(weak_form_inputs)
    u⁰ = Assemblers.get_trial_form(weak_form_inputs)

    JF = - ∫((d(w⁰) ∧ u⁰) ∧ u_h, dΩ)
    lhs_expressions = ((JF,),)

    F = -1/2 * ∫(d(w⁰) ∧ (u_h ∧ u_h), dΩ)
    rhs_expressions = ((F,),)

    weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
    JF, F = Assemblers.assemble(weak_form, bc)

    return JF, F
end

# Solution field for the time-independent problem
weak_form_inputs = Assemblers.WeakFormInputs(V⁰, g)
lhs_expressions, rhs_expressions = Assemblers.L2_projection(weak_form_inputs, dΩ)
weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
A, b = Assemblers.assemble(weak_form)
u_h = A \ b

# fig = Figure()
# ax = Axis(fig[1, 1], title="Newton solution")
# lines!(u_h)
# display(fig)

using LinearAlgebra

function midpoint_newton_step(V, M, K, u_h, dΩ, dt; tolerance=1e-5, max_iterations=50)

    x = u_h
    residual_norms = Float64[]

    weak_form_inputs = Assemblers.WeakFormInputs(V, V)
    bc = Forms.set_dirichlet_boundary_conditions(V, 0.0)

    w⁰ = Assemblers.get_test_form(weak_form_inputs)
    u⁰ = Assemblers.get_trial_form(weak_form_inputs)

    for iteration in 1:max_iterations
        u_mid = 0.5 .* (x .+ u_h)

        x_formfield = Forms.FormField(V⁰, x, "u")
        u_h_formfield = Forms.FormField(V⁰, u_h, "u")

        JF = - (0.5 * ∫((d(w⁰) ∧ u⁰) ∧ u_h_formfield, dΩ) + 0.5 * ∫((d(w⁰) ∧ u⁰) ∧ x_formfield, dΩ))
        lhs_expressions = ((JF,),)

        F = -0.5 * (0.5 * ∫(d(w⁰) ∧ (u_h_formfield ∧ u_h_formfield), dΩ) + 0.5 * ∫(d(w⁰) ∧ (x_formfield ∧ x_formfield), dΩ))
        rhs_expressions = ((F,),)

        weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
        JF, F = Assemblers.assemble(weak_form, bc)

        N = M * (x .- u_h) + dt .* F + dt .* K * u_mid

        push!(residual_norms, norm(N))
        if residual_norms[end] < tolerance
            break
        end

        J = M .+ dt .* JF .+ (0.5*dt) .* K
        x .-= J \ N

    end

    # if residual_norms[end] ≥ tolerance
    #     @warn "Newton did not converge" residual_norms
    # end

    return x, residual_norms
end

function poisson_newton_step(V, M, K, u_h, dΩ, dt; tolerance=1e-5, max_iterations=50)

    x = u_h
    residual_norms = Float64[]

    weak_form_inputs = Assemblers.WeakFormInputs(V, V)
    bc = Forms.set_dirichlet_boundary_conditions(V, 0.0)

    w⁰ = Assemblers.get_test_form(weak_form_inputs)
    u⁰ = Assemblers.get_trial_form(weak_form_inputs)

    for iteration in 1:max_iterations
        u_mid = 0.5 .* (x .+ u_h)

        x_formfield = Forms.FormField(V⁰, x, "u")
        u_h_formfield = Forms.FormField(V⁰, u_h, "u")

        JF = - (1/2 * ∫((d(w⁰) ∧ u⁰) ∧ u_h_formfield, dΩ) + 1/2 * ∫((d(w⁰) ∧ u⁰) ∧ x_formfield, dΩ)) # ?
        lhs_expressions = ((JF,),)

        F = -0.5 * (1/3 * ∫(d(w⁰) ∧ (u_h_formfield ∧ u_h_formfield), dΩ) + 1/3 * ∫(d(w⁰) ∧ (u_h_formfield ∧ x_formfield), dΩ) + 1/3 * ∫(d(w⁰) ∧ (x_formfield ∧ x_formfield), dΩ))
        rhs_expressions = ((F,),)

        weak_form = Assemblers.WeakForm(lhs_expressions, rhs_expressions, weak_form_inputs)
        JF, F = Assemblers.assemble(weak_form, bc)

        N = M * (x .- u_h) + dt .* F + dt .* K * u_mid

        push!(residual_norms, norm(N))
        if residual_norms[end] < tolerance
            break
        end

        J = M .+ dt .* JF .+ (0.5*dt) .* K
        x .-= J \ N

    end

    # if residual_norms[end] ≥ tolerance
    #     @warn "Newton did not converge" residual_norms
    # end

    return x, residual_norms
end

dt = 0.0001
num_steps = 101
history = (
        t=[0.0],
        newton_residuals=Vector{Float64}[],
    )

snapshots = Dict{Int, NamedTuple}()
snapshot_steps = 1:num_steps
0 in snapshot_steps && (snapshots[0] = (; t=0.0, u_h=copy(u_h)))
for step in 1:num_steps
    u_h, residual_norms = poisson_newton_step(V⁰, M, K, u_h, dΩ, dt)
    push!(history.t, step * dt)
    push!(history.newton_residuals, residual_norms)
    if step in snapshot_steps
        snapshots[step] = (; t=step * dt, u_h=copy(u_h))
    end
end

fig = Figure()
ax = Axis(
    fig[1, 1];
    xlabel="x",
    ylabel="u",
)
u_plot = Observable(snapshots[1].u_h)
lines!(ax, u_plot)
record(
    fig,
    "burgers_1d.mp4",
    1:num_steps;
    framerate=30,
) do step
    u_plot[] = snapshots[step].u_h
end

# fig = Figure()
# ax = Axis(fig[1, 1], title="Newton solution")
# lines!(x)
# display(fig)

# fig = Figure()
# ax = Axis(fig[1, 1], xscale=log10, yscale=log10, title="Residual norms")
# lines!(residual_norms)
# display(fig)

"burgers_1d.mp4"